<center>
<h1 style="font-family:verdana">
 🏘 Sistema de diàleg basat en regles 🏘 <h1>








<p>   🎯 <b>Objectiu</b>: en aquesta pràctica aprendrem a crear un assistent senzill de compra d'habitatges. El sistema haurà d'identificar i mostrar les cases que coincideixen amb les preferències de l'usuari. A poc a poc anirem afegint-hi funcionalitats, perquè el sistema siga més complet. </p>

<p> ✨ <b>Contingut</b>: en primer lloc, començarem amb un exemple senzill en què el sistema llançarà unes preguntes i respostes predefinides i l'usuari haurà d'escollir. A poc a poc anirem afegint-hi funcionalitats, perquè el sistema siga més complet. </p>

✏ <b>Exercicis</b>: en cada secció anireu trobant exercicis que haureu d'anar resolent.

---

<h2> Índex </h2>


1. [Fitxer JSON](#section-one)
  * [Exercici 1](#ex-one)
2. [Sistema de diàleg senzill](#section-two)
  * [Exercici 2](#ex-two)
  * [Exercici 3](#ex-three)
  * [Exercici 4](#ex-four)
3. [Millorem el sistema de diàleg](#section-three)
  * [Exercici 5](#ex-five)
  * [Exercici 6](#ex-six)
4. [Lliurable](#section-four)
---


In [1]:
# Si s'executa a Colab, muntem Drive; si no (execució local), llegim el JSON del directori actual.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_PATH = '/content/drive/MyDrive/TVD/house_data.json'
except ImportError:
    DATA_PATH = 'house_data.json'

In [2]:
from nltk.tokenize.treebank import TreebankWordTokenizer
import nltk
nltk.download('omw-1.4')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import TweetTokenizer

import json
import sys

[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\berta\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


<h1><a name="section-one"> 1. Fitxer JSON </a></h1>

En primer lloc, carregarem el fitxer JSON per analitzar-ne l'estructura.

In [4]:
with open(DATA_PATH) as f:
  data = json.load(f)

print('Start message: ', data['start_message'])
print('End message: ', data['end_message'])
print('Questions: ', data['questions'])
print('House: ', data['houses'])

Start message:  Welcome to the House Buying Assistant!
End message:  Thank you for using the House Buying Assistant. Goodbye!
Questions:  [{'question': 'How many bedrooms do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (1 - 5 bedrooms): ', 'answer_key': 'bedrooms'}, {'question': 'How many bathrooms do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (1 - 3 bathrooms): ', 'answer_key': 'bathrooms'}, {'question': 'What is your budget for the house?', 'type': 'numerical', 'prompt': 'Enter your choice (1k - 999k euros):', 'answer_key': 'price'}, {'question': 'How many square meters do you need?', 'type': 'numerical', 'prompt': 'Enter your choice (30 - 200 meters)', 'answer_key': 'square_meters'}, {'question': 'Which city or neighborhood would you prefer?', 'type': 'multichoice', 'prompt': 'Enter your choice', 'answer_key': 'location'}]
House:  [{'id': 1, 'type': 'rent', 'bedrooms': '3', 'bathrooms': '2', 'price': '1200', 'square_meters': '100', 'floor': '3', '

Podeu veure com al fitxer JSON podem trobar el missatge de benvinguda i el missatge de comiat. Les diferents preguntes predefinides que el sistema demanarà a l'usuari. I les cases que l'agència té disponibles per a oferir a l'usuari.

Si no acabeu d'entendre el format, podeu copiar el contingut del fitxer JSON [aquí](https://jsonviewer.stack.hu/) per visualitzar millor l'estructura.

---



 <h1><a name="ex-one"><center> ✏ Exercici 1 ✏</a></h1>


En aquest primer exercici us demanem que afegiu una nova casa manualment al fitxer JSON amb l'id 26. La resta de camps els podeu emplenar com vulgueu.

Si ho heu fet correctament podreu veure al següent *print* les dades que heu introduït.

**Solució.** Hem afegit la casa manualment al final de la llista `houses` del fitxer `house_data.json`:

```json
{
  "id": 26,
  "type": "rent",
  "bedrooms": "2",
  "bathrooms": "1",
  "price": "950",
  "square_meters": "70",
  "floor": "5",
  "elevator": "Yes",
  "commercial_use": "No",
  "terrace": "Yes",
  "location": "Barcelona"
}

In [5]:
id = 26
house = data['houses'][id-1]

print(f"House ID: {house['id']}")
print(f"Bedrooms: {house['bedrooms']}")
print(f"Bathrooms: {house['bathrooms']}")
print(f"Price: {house['price']}")
print(f"Square Meters: {house['square_meters']}")
print(f"Location: {house['location']}")

House ID: 26
Bedrooms: 2
Bathrooms: 1
Price: 950
Square Meters: 70
Location: Barcelona


---

<h1><a name="section-two"> 2. Sistema de diàleg senzill </a></h1>


En aquesta primera part crearem un sistema senzill que mostrarà les preguntes predefinides i, a continuació l'usuari haurà d'introduir la resposta. Tractarem de manera diferent les preguntes en què la resposta siga un número i les preguntes d'opció múltiple.

Intenteu entendre el funcionament, ja que treballarem sobre aquest codi.

In [6]:
def print_question(prompt, possible_options = []):
    print(prompt)
    if not len(possible_options) == 0:
      print("Options:", ", ".join(possible_options))

def initialize_available_options(house_data, available_options):
    for house in house_data['houses']:
        for key, value in house.items():
            available_options.setdefault(key, set()).add(value)

def preprocess_answer(answer):
    answer = nltk.word_tokenize(answer)
    return answer

def get_numerical_value(tok_answer):
    for token in tok_answer:
        if token.isnumeric() or token[:-1].isnumeric():
            return token
    return ''

#Exemple de frase tokenitzada
print(preprocess_answer('Vull un pis de 35k €'))

['Vull', 'un', 'pis', 'de', '35k', '€']


In [7]:
def process_numerical_question(question):
    print_question(question['question'])
    while True:
      answer = input(question['prompt'])
      tok_answer = preprocess_answer(answer)
      value = get_numerical_value(tok_answer)
      if not value == '':
        return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
      answer = input(question['prompt'])
      if answer in options:
        return answer

user_preferences, available_options = {}, {}
initialize_available_options(data, available_options)

for question in data['questions']:
  answer_key = question['answer_key']
  possible_options = list(available_options.get(answer_key))

  if question['type'] == 'numerical':
    answer = process_numerical_question(question)
  else:
    answer = process_multichoice_question(question, possible_options)

  user_preferences[answer_key] = answer

How many bedrooms do you need?


How many bathrooms do you need?
What is your budget for the house?
How many square meters do you need?
Which city or neighborhood would you prefer?
Options: Barcelona, Esplugues de Llobregat, Santa Coloma de Gramenet, L'Hospitalet de Llobregat




---


 <h1><a name="ex-two"><center> ✏ Exercici 2 ✏ </a></h1>

A continuació millorarem un poc el sistema. Us proposem que afegiu la frase de benvinguda que tenim guardada al fitxer JSON. És a dir, abans de mostrar la primera pregunta el sistema ens donarà la benvinguda.

In [14]:
def dialogue_ex2(data):
    print(data['start_message'])                 
    user_preferences, available_options = {}, {}
    initialize_available_options(data, available_options)

    for question in data['questions']:
        answer_key = question['answer_key']
        possible_options = sorted(available_options.get(answer_key))
        if question['type'] == 'numerical':
            answer = process_numerical_question(question)
        else:
            answer = process_multichoice_question(question, possible_options)
        user_preferences[answer_key] = answer
    return user_preferences

user_preferences = dialogue_ex2(data)
user_preferences

Welcome to the House Buying Assistant!
How many bedrooms do you need?
How many bathrooms do you need?
What is your budget for the house?
How many square meters do you need?
Which city or neighborhood would you prefer?
Options: Barcelona, Esplugues de Llobregat, L'Hospitalet de Llobregat, Santa Coloma de Gramenet


{'bedrooms': '2',
 'bathrooms': '1',
 'price': '160k',
 'square_meters': '80',
 'location': 'Santa Coloma de Gramenet'}

---


 <h1><a name="ex-three"><center> ✏ Exercici 3 ✏ </a></h1>

En aquest exercici us demanem que permeteu a l'usuari abandonar el programa quan ho desitge. Per exemple, si l'usuari escriu `quit`, que el sistema s'acomiade amb l'oració de comiat que trobareu al JSON i finalitze el programa.


🙃 `sys.exit()` genera una excepció SystemExit.

In [9]:
QUIT_WORDS = {'quit'}

def check_quit(answer):
    """Si l'usuari escriu 'quit' s'acomiada amb el missatge del JSON i finalitza (SystemExit)."""
    if answer.strip().lower() in QUIT_WORDS:     # .lower() -> també funciona amb 'Quit' o 'QUIT'
        print(data['end_message'])
        sys.exit()

def process_numerical_question(question):
    print_question(question['question'])
    while True:
        answer = input(question['prompt'])
        check_quit(answer)
        value = get_numerical_value(preprocess_answer(answer))
        if value != '':
            return value

def process_multichoice_question(question, options):
    print_question(question['question'], options)
    while True:
        answer = input(question['prompt'])
        check_quit(answer)
        if answer in options:
            return answer

def run_dialogue(dialogue_fn, *args):
    """Executa un diàleg capturant el SystemExit, perquè a Colab la cel·la acabi netament
    (sense el traceback que mostra Jupyter quan es crida sys.exit())."""
    try:
        return dialogue_fn(*args)
    except SystemExit:
        return None

# Prova: l'usuari surt a la segona pregunta escrivint 'Quit'
prefs_ex3 = run_dialogue(dialogue_ex2, data)
print('Preferències recollides:', prefs_ex3)

Welcome to the House Buying Assistant!
How many bedrooms do you need?
Thank you for using the House Buying Assistant. Goodbye!
Preferències recollides: None


Funciona també si escriviu `Quit`?

**Resposta.** Sí: abans de comparar normalitzem la resposta amb `answer.strip().lower()`, de manera que `quit`, `Quit`, `QUIT` o `quit ` (amb espais) funcionen igual. Si haguéssim comparat directament `answer == 'quit'`, `Quit` no funcionaria. Encara hi ha altres maneres naturals de voler sortir (`exit`, `adéu`, `sortir`...) que el sistema no entén: és un dels casos límit que resolem a l'exercici 7.

---

 <h1><a name="ex-four"><center> ✏ Exercici 4 ✏ </a></h1>

Fins ara tenim a `user_preferences` les opcions que l'usuari ha escollit.







In [10]:
user_preferences

{'bedrooms': '2',
 'bathrooms': '1',
 'price': '50k',
 'square_meters': '100',
 'location': 'Esplugues de Llobregat'}

Per tant, ara haurem de revisar si l'agència disposa d'alguna casa disponible amb aquestes característiques. Per això us proposem completar la funció `find_suitable_houses`. Aquesta funció tornarà una llista amb tants diccionaris com cases disponibles. Per exemple si segons les preferències de l'usuari només està disponible la casa amb id 3, `suitable_house` serà:


```
[{'id': 3, 'bedrooms': '2', 'bathrooms': '1', 'price': '160k', 'square_meters': '80', 'location': 'Santa Coloma de Gramenet'}]
```

**Decisions de disseny del filtratge.** Interpretem les preferències com a *requisits mínims o màxims*, que és com les entendria un agent immobiliari:

| Clau | Criteri | Motiu |
|---|---|---|
| `bedrooms`, `bathrooms`, `square_meters` | casa ≥ valor demanat | qui necessita 2 habitacions també accepta un pis de 3 |
| `price` | casa ≤ pressupost | el pressupost és un màxim |
| `location` | igualtat | l'usuari tria una zona de la llista |

Els preus del JSON tenen formats diferents (`"250k"` en venda, `"1200"` en lloguer), així que els convertim a euros amb `parse_price` abans de comparar. Les preferències amb valor `None` (exercici 5) s'ignoren.

In [11]:
def parse_price(value):
    """Converteix un preu o número a float: '250k' -> 250000.0, '1200' -> 1200.0, '900€' -> 900.0"""
    s = str(value).strip().lower().replace('€', '')
    return float(s[:-1]) * 1000 if s.endswith('k') else float(s)

# 'min': el valor de la casa ha de ser >= al demanat | 'max': <= | 'eq': igual
FILTER_RULES = {'bedrooms': 'min', 'bathrooms': 'min', 'square_meters': 'min',
                'price': 'max', 'location': 'eq'}

def house_matches(house, user_preferences, rules=FILTER_RULES):
    for key, wanted in user_preferences.items():
        if wanted is None or key not in rules:          # None = sense preferència
            continue
        rule = rules[key]
        if rule == 'eq' and house[key] != wanted:
            return False
        if rule == 'min' and parse_price(house[key]) < parse_price(wanted):
            return False
        if rule == 'max' and parse_price(house[key]) > parse_price(wanted):
            return False
    return True

def find_suitable_houses(data, user_preferences):
    suitable_houses = [house for house in data['houses'] if house_matches(house, user_preferences)]
    return suitable_houses

---



Una vegada tingueu la funció `find_suitable_houses` programada, podreu executar el següent *script* que us mostrarà, si n'hi ha, la casa o cases disponibles amb les característiques que heu triat.

In [15]:
def print_suitable_houses(suitable_houses):
  if suitable_houses:
    print("\nBased on your preferences, the most suitable houses are:")
    for house in suitable_houses:
      print(f"House ID: {house['id']}")
      print(f"Type: {house['type']}")
      print("Bedrooms:", house['bedrooms'])
      print("Bathrooms:", house['bathrooms'])
      print("Price:", house['price'], "euros")
      print("Square Meters:", house['square_meters'], "m^2")
      print("Location:", house['location'])
      print()
  else:
    print("\nSorry, no suitable houses match your preferences. \n")


suitable_houses = find_suitable_houses(data, user_preferences)
print_suitable_houses(suitable_houses)


Based on your preferences, the most suitable houses are:
House ID: 1
Type: rent
Bedrooms: 3
Bathrooms: 2
Price: 1200 euros
Square Meters: 100 m^2
Location: Santa Coloma de Gramenet

House ID: 3
Type: rent
Bedrooms: 2
Bathrooms: 1
Price: 800 euros
Square Meters: 80 m^2
Location: Santa Coloma de Gramenet

House ID: 6
Type: sale
Bedrooms: 2
Bathrooms: 1
Price: 120k euros
Square Meters: 80 m^2
Location: Santa Coloma de Gramenet

House ID: 7
Type: rent
Bedrooms: 5
Bathrooms: 2
Price: 2000 euros
Square Meters: 150 m^2
Location: Santa Coloma de Gramenet



<h1><a name="section-three"> 3. Millorem el sistema de diàleg </a></h1>


---


 <h1><a name="ex-five"><center> ✏ Exercici 5 ✏ </a></h1>

Tal com haureu comprovat sempre esteu obligats a triar una opció. En aquest últim exercici proposem que afegiu l'opció de no triar-ne cap. És a dir, si ens és igual el nombre d'habitacions, podrem posar per exemple `any` i el sistema ens farà la següent pregunta.

In [16]:
ANY_WORDS = {'any'}

def is_any(answer):
    return answer.strip().lower() in ANY_WORDS

def process_numerical_question(question):
    print_question(question['question'] + "  (type 'any' if you don't mind)")
    while True:
        answer = input(question['prompt'])
        check_quit(answer)
        if is_any(answer):
            return None                                 # None = sense preferència
        value = get_numerical_value(preprocess_answer(answer))
        if value != '':
            return value

def process_multichoice_question(question, options):
    print_question(question['question'] + "  (type 'any' if you don't mind)", options)
    while True:
        answer = input(question['prompt'])
        check_quit(answer)
        if is_any(answer):
            return None
        if answer in options:
            return answer

# find_suitable_houses ja ignora les claus amb valor None, així que no cal modificar-la.
user_preferences = run_dialogue(dialogue_ex2, data)
print('\nPreferències:', user_preferences)
print_suitable_houses(find_suitable_houses(data, user_preferences))

Welcome to the House Buying Assistant!
How many bedrooms do you need?  (type 'any' if you don't mind)
How many bathrooms do you need?  (type 'any' if you don't mind)
What is your budget for the house?  (type 'any' if you don't mind)
How many square meters do you need?  (type 'any' if you don't mind)
Which city or neighborhood would you prefer?  (type 'any' if you don't mind)
Options: Barcelona, Esplugues de Llobregat, L'Hospitalet de Llobregat, Santa Coloma de Gramenet

Preferències: {'bedrooms': None, 'bathrooms': None, 'price': None, 'square_meters': None, 'location': None}

Based on your preferences, the most suitable houses are:
House ID: 1
Type: rent
Bedrooms: 3
Bathrooms: 2
Price: 1200 euros
Square Meters: 100 m^2
Location: Santa Coloma de Gramenet

House ID: 2
Type: sale
Bedrooms: 4
Bathrooms: 2
Price: 250k euros
Square Meters: 120 m^2
Location: Santa Coloma de Gramenet

House ID: 3
Type: rent
Bedrooms: 2
Bathrooms: 1
Price: 800 euros
Square Meters: 80 m^2
Location: Santa Coloma

---



 <h1><a name="ex-six"><center> ✏ Exercici 6 ✏ </a></h1>

Amplieu el vostre xatbot perquè tinga en compte la següent informació sobre:

1. **Tipus d'habitatge:** Pregunteu a l'usuari si vol una casa per a compra (tipus *"sale"* a les dades) o per llogar (tipus *"rent"*). Al final de l'execució, el sistema haurá de mostrar habitatges amb el tipus seleccionat.

2. **Ingressos:** S'aconsella no dedicar més del 35% dels ingressos a l'habitatge. En cas d'escollir lloguer, preguntar a l'usuari els ingressos mensuals de la seua llar, i només mostrar aquells que estiguen per sota del 35%.

3. **Planta:** Preguntar als usuaris a quina planta volen viure. Al final de l'execució, només s'oferiràn habitatges que estiguen en aquella planta o superior.

4. **Terrassa:** Preguntar a l'usuari si vol un habitatge amb terrassa o no. En cas de voler terrassa, només s'oferiran habitatges amb terrassa. En cas contrari, oferir habitatges amb terrassa o sense.

5. **Ascensor:** Preguntar a l'usuari si vol un habitatge amb ascensor o no. En cas de voler ascensor, només s'oferiran habitatges amb ascensor. En cas contrari, oferir habitatges amb ascensor o sense.

6. **Us comercial:** Preguntar a l'usuari si vol fer servir l'habitatge com a negoci. En cas afirmatiu, només oferir habitatges amb la propietat *"commercial use"*

---





**Solució.** Afegim les preguntes específiques de l'exercici 6 al camp `questions_ex6` del fitxer JSON. Hi ha dos elements nous:
* un tipus de pregunta `yesno` (resposta `yes`/`no`, o `any`), i
* un camp opcional `condition`: la pregunta de pressupost només es fa si l'usuari ha triat comprar (`type == 'sale'`) i la d'ingressos només si ha triat llogar (`type == 'rent'`).

In [17]:
QUESTIONS_EX6_JSON = data['questions_ex6']

# Combina les preguntes predefinides amb les preguntes específiques de l'exercici 6.
QUESTIONS_EX6 = ([QUESTIONS_EX6_JSON[0]] + data['questions'][:2]
                 + QUESTIONS_EX6_JSON[1:3] + data['questions'][3:]
                 + QUESTIONS_EX6_JSON[3:])

def process_yesno_question(question):
    print_question(question['question'] + "  (type 'any' if you don't mind)", ['yes', 'no'])
    while True:
        answer = input(question['prompt'])
        check_quit(answer)
        if is_any(answer):
            return None
        if answer.strip().lower() in ('yes', 'no'):
            return answer.strip().capitalize()                   # mateix format que el JSON

def dialogue_ex6(data, questions=QUESTIONS_EX6):
    print(data['start_message'], flush=True)
    user_preferences, available_options = {}, {}
    initialize_available_options(data, available_options)
    for question in questions:
        if 'condition' in question:                              # preguntes condicionals
            condition = question['condition']
            key, value = condition['key'], condition['value']
            if user_preferences.get(key) != value:
                continue
        answer_key = question['answer_key']
        if question['type'] == 'numerical':
            answer = process_numerical_question(question)
        elif question['type'] == 'yesno':
            answer = process_yesno_question(question)
        else:
            answer = process_multichoice_question(question, sorted(available_options.get(answer_key)))
        user_preferences[answer_key] = answer
    return user_preferences

In [18]:
FILTER_RULES_EX6 = dict(FILTER_RULES, type='eq', floor='min')   # tipus exacte; planta >= demanada
MAX_RENT_SHARE = 0.35                                           # màxim 35 % dels ingressos

def find_suitable_houses(data, user_preferences):
    suitable_houses = []
    for house in data['houses']:
        if not house_matches(house, user_preferences, FILTER_RULES_EX6):
            continue
        # Ingressos (només lloguer): el lloguer no pot superar el 35 % dels ingressos mensuals
        income = user_preferences.get('income')
        if user_preferences.get('type') == 'rent' and income is not None \
                and parse_price(house['price']) > MAX_RENT_SHARE * parse_price(income):
            continue
        # Terrassa, ascensor i ús comercial: només filtrem si l'usuari respon 'Yes'
        if any(user_preferences.get(k) == 'Yes' and house[k] != 'Yes'
               for k in ('terrace', 'elevator', 'commercial_use')):
            continue
        suitable_houses.append(house)
    return suitable_houses

def print_suitable_houses(suitable_houses):
    if suitable_houses:
        print("\nBased on your preferences, the most suitable houses are:")
        for house in suitable_houses:
            print(f"House ID: {house['id']}")
            print(f"Type: {house['type']}")
            print("Bedrooms:", house['bedrooms'], "| Bathrooms:", house['bathrooms'])
            print("Price:", house['price'], "euros")
            print("Square Meters:", house['square_meters'], "m^2")
            print("Floor:", house['floor'], "| Elevator:", house['elevator'],
                  "| Terrace:", house['terrace'], "| Commercial use:", house['commercial_use'])
            print("Location:", house['location'])
            print()
    else:
        print("\nSorry, no suitable houses match your preferences. \n")

In [19]:
user_preferences = run_dialogue(dialogue_ex6, data)
print('\nPreferències:', user_preferences)
if user_preferences is not None:
    print_suitable_houses(find_suitable_houses(data, user_preferences))

Welcome to the House Buying Assistant!
Do you want to buy (sale) or to rent?  (type 'any' if you don't mind)
Options: rent, sale
How many bedrooms do you need?  (type 'any' if you don't mind)
How many bathrooms do you need?  (type 'any' if you don't mind)
What is your household's monthly income?  (type 'any' if you don't mind)
How many square meters do you need?  (type 'any' if you don't mind)
Which city or neighborhood would you prefer?  (type 'any' if you don't mind)
Options: Barcelona, Esplugues de Llobregat, L'Hospitalet de Llobregat, Santa Coloma de Gramenet
From which floor would you like to live? (0 = ground floor)  (type 'any' if you don't mind)
Do you want a terrace?  (type 'any' if you don't mind)
Options: yes, no
Do you want an elevator?  (type 'any' if you don't mind)
Options: yes, no
Do you want to use the house for business (commercial use)?  (type 'any' if you don't mind)
Options: yes, no

Preferències: {'type': 'rent', 'bedrooms': None, 'bathrooms': '1', 'income': '10k'

 <h1><a name="ex-seven"><center> ✏ Exercici 7 ✏ </a></h1>

Ara ja teniu un sistema funcional. Aplicant el mètode vist a classe (Exemples, Camins, Prototipat i Proves) milloreu aquest sistema per donar una millor experiència als usuaris.

---